In [1]:
import sys
sys.path.append('/Users/joachim/texjs/lva/IntroSC/ASC-ODE/build/mechsystem')
sys.path.append('../build/mechsystem')

# Force reimport of the module
import importlib
if 'mass_spring' in sys.modules:
    del sys.modules['mass_spring']

from mass_spring import *
from pythreejs import *

In [ ]:
mss = MassSpringSystem3d()
mss.gravity = (0,0,-9.81)

mA = mss.add (Mass(1, (1,0,0)))
mB = mss.add (Mass(2, (2,0,0)))
f1 = mss.add (Fix( (0,0,0)) )
mss.add_constraint (DistanceConstraint(f1, mA, 1.0))
mss.add_constraint (DistanceConstraint(mA, mB, 1.0))

In [ ]:
masses = []
for m in mss.masses:
    masses.append(
        Mesh(SphereBufferGeometry(0.2, 16, 16),
             MeshStandardMaterial(color='red'),
             position=m.pos)) 

fixes = []
for f in mss.fixes:
    fixes.append(
        Mesh(SphereBufferGeometry(0.2, 32, 16),
             MeshStandardMaterial(color='blue'),
             position=f.pos)) 

constraintpos = []
for c in mss.constraints:
    pA = mss[c.connectors[0]].pos
    pB = mss[c.connectors[1]].pos
    constraintpos.append ([ pA, pB ] ) 

springgeo = LineSegmentsGeometry(positions=constraintpos)
m2 = LineMaterial(linewidth=3, color='cyan')
springs = LineSegments2(springgeo, m2)    

axes = AxesHelper(1)

In [ ]:
view_width = 600
view_height = 400

camera = PerspectiveCamera( position=[10, 6, 10], aspect=view_width/view_height)
key_light = DirectionalLight(position=[0, 10, 10])
ambient_light = AmbientLight()

scene = Scene(children=[*masses, *fixes, springs, axes, camera, key_light, ambient_light])
controller = OrbitControls(controlling=camera)
renderer = Renderer(camera=camera, scene=scene, controls=[controller],
                    width=view_width, height=view_height)

renderer

In [ ]:
from time import sleep
for i in range(10000):
    mss.simulate (0.02, 100)
    for m,mvis in zip(mss.masses, masses):
        mvis.position = (m.pos[0], m.pos[1], m.pos[2])

    constraintpos = []
    for c in mss.constraints:
        pA = mss[c.connectors[0]].pos
        pB = mss[c.connectors[1]].pos
        constraintpos.append ([ pA, pB ]) 
    springs.geometry = LineSegmentsGeometry(positions=constraintpos)
    sleep(0.01)

## 5-Point Pendulum Simulation

Now let's create a similar system with 5 masses (points):

In [ ]:
mss5 = MassSpringSystem3d()
mss5.gravity = (0,0,-9.81)

m1 = mss5.add (Mass(1, (1,0,0)))
m2 = mss5.add (Mass(2, (2,0,0)))
m3 = mss5.add (Mass(3, (3,0,0)))
m4 = mss5.add (Mass(4, (4,0,0)))
m5 = mss5.add (Mass(5, (5,0,0)))
f1_5pt = mss5.add (Fix( (0,0,0)) )
mss5.add_constraint (DistanceConstraint(f1_5pt, m1, 1.0))
mss5.add_constraint (DistanceConstraint(m1, m2, 1.0))
mss5.add_constraint (DistanceConstraint(m2, m3, 1.0))
mss5.add_constraint (DistanceConstraint(m3, m4, 1.0))
mss5.add_constraint (DistanceConstraint(m4, m5, 1.0))

In [ ]:
masses5 = []
for m in mss5.masses:
    masses5.append(
        Mesh(SphereBufferGeometry(0.2, 16, 16),
             MeshStandardMaterial(color='red'),
             position=m.pos)) 

fixes5 = []
for f in mss5.fixes:
    fixes5.append(
        Mesh(SphereBufferGeometry(0.2, 32, 16),
             MeshStandardMaterial(color='blue'),
             position=f.pos)) 

constraintpos5 = []
for c in mss5.constraints:
    pA = mss5[c.connectors[0]].pos
    pB = mss5[c.connectors[1]].pos
    constraintpos5.append ([ pA, pB ] ) 

springgeo5 = LineSegmentsGeometry(positions=constraintpos5)
m2_5 = LineMaterial(linewidth=3, color='cyan')
springs5 = LineSegments2(springgeo5, m2_5)    

axes5 = AxesHelper(1)

In [ ]:
view_width5 = 600
view_height5 = 400

camera5 = PerspectiveCamera( position=[10, 6, 10], aspect=view_width5/view_height5)
key_light5 = DirectionalLight(position=[0, 10, 10])
ambient_light5 = AmbientLight()

scene5 = Scene(children=[*masses5, *fixes5, springs5, axes5, camera5, key_light5, ambient_light5])
controller5 = OrbitControls(controlling=camera5)
renderer5 = Renderer(camera=camera5, scene=scene5, controls=[controller5],
                    width=view_width5, height=view_height5)

renderer5

In [ ]:
from time import sleep
for i in range(10000):
    mss5.simulate (0.02, 100)
    for m,mvis in zip(mss5.masses, masses5):
        mvis.position = (m.pos[0], m.pos[1], m.pos[2])

    constraintpos5 = []
    for c in mss5.constraints:
        pA = mss5[c.connectors[0]].pos
        pB = mss5[c.connectors[1]].pos
        constraintpos5.append ([ pA, pB ]) 
    springs5.geometry = LineSegmentsGeometry(positions=constraintpos5)
    sleep(0.01)

## Spinning Top (Kreisel)

A spinning top is a rigid body structure that maintains its shape through distance constraints while rotating with angular momentum:


In [2]:
# Create a spinning top with 3 masses in a triangular configuration
mss_top = MassSpringSystem3d()
mss_top.gravity = (0, 0, -9.81)

# Pivot point at origin
f_pivot = mss_top.add(Fix((0, 0, 0)))

# Three masses arranged in a triangle in the xy-plane, offset down in z
# They form an equilateral triangle with side length 1.0
import math
r = 0.5  # radius from center to each mass in xy-plane
pivot_dist = 1.0  # distance from pivot to each mass

# Calculate z-offset so that distance from origin = pivot_dist
# sqrt(r^2 + z^2) = pivot_dist  =>  z = -sqrt(pivot_dist^2 - r^2)
z_offset = -math.sqrt(pivot_dist**2 - r**2)
mass_val = 1.0

# Mass positions (equilateral triangle centered at (0, 0, z_offset))
# Initial positions satisfy the pivot distance constraint exactly
m_top_1 = mss_top.add(Mass(mass_val, (r, 0, z_offset)))
m_top_2 = mss_top.add(Mass(mass_val, (r*math.cos(2*math.pi/3), r*math.sin(2*math.pi/3), z_offset)))
m_top_3 = mss_top.add(Mass(mass_val, (r*math.cos(4*math.pi/3), r*math.sin(4*math.pi/3), z_offset)))

# Distance constraints: connect pivot to each mass
mss_top.add_constraint(DistanceConstraint(f_pivot, m_top_1, pivot_dist))
mss_top.add_constraint(DistanceConstraint(f_pivot, m_top_2, pivot_dist))
mss_top.add_constraint(DistanceConstraint(f_pivot, m_top_3, pivot_dist))

# Distance constraints: connect masses to each other (rigid triangle)
triangle_dist = math.sqrt(3) * r
mss_top.add_constraint(DistanceConstraint(m_top_1, m_top_2, triangle_dist))
mss_top.add_constraint(DistanceConstraint(m_top_2, m_top_3, triangle_dist))
mss_top.add_constraint(DistanceConstraint(m_top_3, m_top_1, triangle_dist))

# Initialize rotational velocities (angular velocity around z-axis)
omega = 10.0  # rad/s
# For rotation around z: velocity = omega × position = (-omega*y, omega*x, 0)
for i, m in enumerate(mss_top.masses):
    pos = m.pos
    mss_top.set_mass_velocity(i, (-omega * pos[1], omega * pos[0], 0.0))

print("✓ Spinning top created with 3 masses")
print(f"  - 1 pivot point fixed at origin")
print(f"  - 3 masses at radius {r} m in xy-plane, z-offset = {z_offset:.3f} m")
print(f"  - Initial angular velocity: {omega} rad/s around z-axis")
print(f"  - 6 distance constraints maintaining rigid body shape")
print(f"\nConstraints maintain:")
print(f"  - Each mass at distance {pivot_dist:.3f} m from pivot")
print(f"  - Triangle edge length {triangle_dist:.3f} m")

✓ Spinning top created with 3 masses
  - 1 pivot point fixed at origin
  - 3 masses at radius 0.5 m in xy-plane, z-offset = -0.866 m
  - Initial angular velocity: 10.0 rad/s around z-axis
  - 6 distance constraints maintaining rigid body shape

Constraints maintain:
  - Each mass at distance 1.000 m from pivot
  - Triangle edge length 0.866 m


In [3]:
# Visualize the spinning top
masses_top = []
for m in mss_top.masses:
    masses_top.append(
        Mesh(SphereBufferGeometry(0.1, 16, 16),
             MeshStandardMaterial(color='red'),
             position=m.pos))

fixes_top = []
for f in mss_top.fixes:
    fixes_top.append(
        Mesh(SphereBufferGeometry(0.15, 32, 16),
             MeshStandardMaterial(color='blue'),
             position=f.pos))

constraintpos_top = []
for c in mss_top.constraints:
    pA = mss_top[c.connectors[0]].pos
    pB = mss_top[c.connectors[1]].pos
    constraintpos_top.append([pA, pB])

springgeo_top = LineSegmentsGeometry(positions=constraintpos_top)
m2_top = LineMaterial(linewidth=2, color='cyan')
springs_top = LineSegments2(springgeo_top, m2_top)

axes_top = AxesHelper(1)

In [4]:
# Set up the 3D visualization for the spinning top
view_width_top = 600
view_height_top = 400

camera_top = PerspectiveCamera(position=[2, 2, 2], aspect=view_width_top/view_height_top)
key_light_top = DirectionalLight(position=[0, 10, 10])
ambient_light_top = AmbientLight()

scene_top = Scene(children=[*masses_top, *fixes_top, springs_top, axes_top, camera_top, key_light_top, ambient_light_top])
controller_top = OrbitControls(controlling=camera_top)
renderer_top = Renderer(camera=camera_top, scene=scene_top, controls=[controller_top],
                        width=view_width_top, height=view_height_top)

renderer_top

Renderer(camera=PerspectiveCamera(aspect=1.5, position=(2.0, 2.0, 2.0), projectionMatrix=(1.0, 0.0, 0.0, 0.0, …

In [ ]:
# Animate the spinning top simulation
# Use smaller time steps for numerical stability with multiple constraints
from time import sleep
for i in range(10000):
    mss_top.simulate(0.01, 50)
    for m, mvis in zip(mss_top.masses, masses_top):
        mvis.position = (m.pos[0], m.pos[1], m.pos[2])

    constraintpos_top = []
    for c in mss_top.constraints:
        pA = mss_top[c.connectors[0]].pos
        pB = mss_top[c.connectors[1]].pos
        constraintpos_top.append([pA, pB])
    springs_top.geometry = LineSegmentsGeometry(positions=constraintpos_top)
    sleep(0.01)